In [1]:
!pip install kaggle-benchmarks


In [2]:
import csv
import random

def generate_alien_math_dataset(filename="alien_math_dataset.csv", num_samples=100):
    """
    Generates a dataset of Alien Math equations.
    Rules:
    @ means: (Left * 3) - Right
    # means: Left + (Right * 2)
    Equation format: (A @ B) # C
    """
    data = []
    
    # We loop 100 times to create our dataset
    for _ in range(num_samples):
        # Pick random numbers between 1 and 10 to keep it simple
        A = random.randint(1, 10)
        B = random.randint(1, 10)
        C = random.randint(1, 10)
        
        # Calculate the absolute truth using our "Alien" rules
        # Step 1: resolve the parenthesis (A @ B) -> (A * 3) - B
        step1 = (A * 3) - B
        
        # Step 2: resolve the rest: (step1) # C -> step1 + (C * 2)
        answer = step1 + (C * 2)
        
        # Format it as a string for the AI to read
        equation = f"({A} @ {B}) # {C}"
        
        # Save the pair
        data.append({"equation": equation, "answer": answer})
        
    # Write everything cleanly to a CSV
    with open(filename, mode='w', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=["equation", "answer"])
        writer.writeheader()
        writer.writerows(data)
        
    print(f"Success: {num_samples} alien equations written to {filename}")

if __name__ == "__main__":
    generate_alien_math_dataset()
    

Success: 100 alien equations written to alien_math_dataset.csv


In [3]:
import kaggle_benchmarks as kbench
import pandas as pd
import re

# Load our ground truth dataset
dataset = pd.read_csv("alien_math_dataset.csv")

@kbench.task(name="alien_math_learning")
def alien_math_task(llm, equation: str, answer: str):
    """
    Evaluates if an AI can learn and apply novel mathematical operators on the fly.
    """
    
    prompt = f"""You are a logical reasoning system. You must learn a new mathematical system.
Rules:
1. X @ Y means (X * 3) - Y
2. X # Y means X + (Y * 2)

Solve the following equation step-by-step, but your VERY LAST word must be the final integer answer.
Equation: {equation}
"""
    
    response = llm.prompt(prompt)
    
    # Extract the AI's number
    numbers_found = re.findall(r'-?\d+', response)
    if numbers_found:
        prediction = int(numbers_found[-1])
    else:
        prediction = None
        
    ground_truth = int(answer)

    kbench.assertions.assert_equal(
        prediction, 
        ground_truth, 
        expectation="The AI must correctly calculate the final integer using Alien Math rules."
    )

print("Starting evaluation...")
results = alien_math_task.evaluate(llm=[kbench.llm], evaluation_data=dataset, n_jobs=5)


print("Evaluation Complete! Here is the raw scorecard:")
if hasattr(results, 'to_pandas'):
    display(results.to_pandas())
else:
    print(results)
    


Starting evaluation...


[Parallel(n_jobs=5)]: Using backend ThreadingBackend with 5 concurrent workers.


[Parallel(n_jobs=5)]: Done  22 tasks      | elapsed:   13.3s


[Parallel(n_jobs=5)]: Done  62 tasks      | elapsed:   35.1s


Evaluation Complete! Here is the raw scorecard:
Runs(runs=[Run(task=Task(func=<function alien_math_task at 0x796fccc7dd00>, name='alien_math_learning', description='Evaluates if an AI can learn and apply novel mathematical operators on the fly.', result_type=<class 'kaggle_benchmarks.results.PassFail'>, version=1, store_task=True, store_run=True), result=None, chat=Chat(history=[Message(content='You are a logical reasoning system. You must learn a new mathematical system.\nRules:\n1. X @ Y means (X * 3) - Y\n2. X # Y means X + (Y * 2)\n\nSolve the following equation step-by-step, but your VERY LAST word must be the final integer answer.\nEquation: (6 @ 7) # 7\n', sender=Actor(name='User', avatar='👤'), _status=<Status.SUCCESS: 'success'>, is_visible_to_llm=True, _meta={}), Message(content="Let's solve the equation step-by-step.\n\nThe equation is (6 @ 7) # 7.\n\nFirst, we evaluate the expression inside the parentheses: (6 @ 7).\nAccording to Rule 1: X @ Y means (X * 3) - Y.\nHere, X = 6

[Parallel(n_jobs=5)]: Done 100 out of 100 | elapsed:   56.0s finished
